# Decision Tree, Random Forest, XG Boost #

Welche Faktoren beeinflussen/sagen die Bewertungen von Airbnb-Unterkünften in Berlin vorher?

Zielvariable: review_scores_rating. 

Bei welchen Merkmalen fällt die Zielvariable besonders positiv aus? Welche Eigenschaften einer Airbnb-Unterkunft führen zu hohen Bewertungen? Welche Angabe ist besonders wichtig?

Vorgehen:
1. Kodierungen/Bereinigung der Tabelle

    1.1 alle unnötigen Spalten und Freitextspalten werden gelöscht ("id", "listing_url", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication", "review_scores_location",review_scores_value", "reviews_per_month", "number_of_reviews")

    1.2 Kodierung von instant_bookable in binäres System (true=1, false=0)

    1.3 One-Hot-Encoding für kategoriale Variablen: für neighbourhood_cleansed,neighbourhood_group_cleansed,property_type,room_type wird die Methode One-Hot-Encoding verwendet, womit kategoriale Variablen in numerische Vektoren umzuwandeln. Dabei wird für jede Ausprägung eine eigene Spalte erstellt. Ein Wert von \(1\) signalisiert das Vorhandensein dieser Kategorie, während alle anderen Spalten den Wert \(0\) erhalten. Pro Kategorie wird dann eine neue Variable mit vorheriger Spaltenbezeichnung_Kategorie gebildet.
    
2. Decision Tree

    2.1 Datenvorbereitung

    2.2 Cross-Validation 60-20-20:

        2.2.1 20 % Testdaten abtrennen

        2.2.2 80 % aufteilen in 60/20 für Training und Validation (25 % der Trainingsdaten dann für Validation genutzt)

    2.3 
    
        2.3.1 Training (Decision Tree Regressor: wollen numerische Werte, keine Klasse als Vorhersage/Zielvariable)

        2.3.2 Validierung

        2.3.3 Test

    2.4 Ergebnisse (iterativ)

        2.4.1 Baum visualisieren, Baum als Text

        2.4.2 wichtige Features bestimmen

    Entscheidungen: 
    
    1. Probieren anderer Bäume mittels Veränderungen der Max_depth und min_samples_leaf

    2. Neighbourhood Variablen ignorieren

3. Random Forest

    3.1 Datenvorbereitung

    3.2 Train-Validation-Test-Split 60-20-20:

        3.2.1 20 % Testdaten abtrennen

        3.2.2 80 % aufteilen in 60/20 für Training und Validation (25 % der Trainingsdaten dann für Validation genutzt)

    3.3 

        3.3.1 Training

        3.3.2 Vorhersage erzeugen

        3.3.3 Güte bewerten

    3.4 Ergebnisse:

        3.4.1 wichtige Features bestimmen

4. XG Boost

    4.1 Datenvorbereitung

    4.2 Train-Validation-Test-Split 60-20-20:

        4.2.1 20 % Testdaten abtrennen

        4.2.2 80 % aufteilen in 60/20 für Training und Validation (25 % der Trainingsdaten dann für Validation genutzt)


    4.3

        4.3.1 Training

        4.3.2 Validierung

        4.3.3 Test

    4.4 Ergebnisse:

        4.4.1 wichtige Features bestimmen

5. Ergebnisse insgesamt

6. Nächste Schritte

    6.1 nur auf Nachbarschaften laufen lassen

    6.2 nur auf Nachbarschaften_group laufen lassen

Entscheidung: Splitkriterien für die Bäume: mittlere quadratische Abweichung (erstmal Standardeinstellung) (zur Reduktion der MSE; Weil Zielvariable metrisch ist nur mittlerer oder absoluter Fehler sinnvoll - gehört zu Regressionsbäumen) - Der Baum wählt genau den Split, der die Summe dieser quadrierten Abweichungen über alle resultierenden Knoten hinweg am stärksten minimiert (Varianzreduktion) - berechnet für jeden Teilknoten den Mittelwert der Zielvariable und dann misst er die quadrierten Abweichungen aller Datenpunkte in diesem Knoten von diesem Mittelwert, der minimiert werden soll.

Nicht so robust gegen Ausreißern wie absolute error - daher wird dieser in der nächsten Runde genutzt.

Bisher keine Cross-Validation sondern erstmal nur Train-Validation-Test-Split.


In [2]:
#Importe
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

### 1. Kodierung und Bereinigung der Tabelle ###

In [3]:
#1.1
df = pd.read_csv("listings_clean.csv")

drop_columns = [
    "id",
    "listing_url",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
    "number_of_reviews"
]

df_cleaned = df.drop(columns=drop_columns)

df_cleaned.to_csv("listings_cleaned4Decision.csv", index=False)

print("Neue CSV wurde erstellt: listings_cleaned4Decision.csv")

#1.2 
df = pd.read_csv("listings_cleaned4Decision.csv")

df["instant_bookable"] = df["instant_bookable"].map({
    "t": 1,
    "f": 0
})

#1.3
categorical_columns = [
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "property_type",
    "room_type"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    dtype=int
)


#Speichern
df.to_csv(
    "listings_cleaned4Decision_encoded.csv",
    index=False
)

print("Neue Datei erstellt.")

Neue CSV wurde erstellt: listings_cleaned4Decision.csv
Neue Datei erstellt.


### 2. Decision Tree ###

In [4]:
#2.1
df = pd.read_csv("listings_cleaned4Decision_encoded.csv")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

#2.2
#2.2.1
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#2.2.2
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [5]:
#2.3
#2.3.1 
tree = DecisionTreeRegressor(
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)

tree.fit(X_train, y_train)

#2.3.2
val_predictions = tree.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)

print("Validation R²:", val_r2)

#2.3.3
test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)

print("Test R²:", test_r2)

Validation R²: 0.14632979365603627
Test R²: 0.11903404812973517


In [6]:
#2.4
#2.4.1
#plt.figure(figsize=(30,15))

#plot_tree(
 #   tree,
 #   feature_names=X.columns,
 #   filled=True,
 #   rounded=True,
 #   fontsize=8
#)

#plt.show()

#tree_rules = export_text(
 #   tree,
 #   feature_names=list(X.columns)
#)

#print(tree_rules)

#2.4.2
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                               Feature  Importance
5                       calculated_host_listings_count    0.353073
8                                log_reviews_per_month    0.128592
3                                     availability_365    0.107177
6                                      price_per_guest    0.101034
2                                       minimum_nights    0.087870
1                                                price    0.057209
7                          price_to_neighborhood_ratio    0.040488
4                                     instant_bookable    0.027650
0                                         accommodates    0.020427
171                   property_type_Entire rental unit    0.017808
148                 neighbourhood_group_cleansed_Mitte    0.016593
155    neighbourhood_group_cleansed_Treptow - Köpenick    0.009091
42     neighbourhood_cleansed_Frankfurter Allee Süd FK    0.006292
215                          room_type_Entire home/apt    0.00

Entscheidung 1: Probieren anderer Bäume mittels Veränderungen der Max_depth und min_samples_leaf

In [7]:
#2.3
#2.3.1 
tree = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=16,
    random_state=42
)

tree.fit(X_train, y_train)

#2.3.2
val_predictions = tree.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)

print("Validation R²:", val_r2)

#2.3.3
test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)

print("Test R²:", test_r2)

Validation R²: 0.19686908391874203
Test R²: 0.1689889815843273


In [8]:
#2.4
#2.4.2
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                             Feature  Importance
5                     calculated_host_listings_count    0.553952
2                                     minimum_nights    0.119595
3                                   availability_365    0.106583
8                              log_reviews_per_month    0.090765
6                                    price_per_guest    0.089315
1                                              price    0.028076
4                                   instant_bookable    0.011714
0                                       accommodates    0.000000
7                        price_to_neighborhood_ratio    0.000000
9                   neighbourhood_cleansed_Adlershof    0.000000
10               neighbourhood_cleansed_Albrechtstr.    0.000000
11             neighbourhood_cleansed_Alexanderplatz    0.000000
12            neighbourhood_cleansed_Allende-Viertel    0.000000
13               neighbourhood_cleansed_Alt  Treptow    0.000000
14  neighbourhood_cleanse

Entscheidung: Wichtige Features calculated_host_listings_count (Nummer aktiver Listings pro Host), log_reviews_per_month (Nummer der Reviews pro Objekt und Monat) herausnehmen, da sie vom Host nicht direkt beeinflusst werden können.
Dennoch wichtiges Ergebnis 

In [9]:
#1.1
df = pd.read_csv("listings_cleaned4Decision_encoded.csv")
df_ohne_host_reviews = df.drop(
    columns=[
        "calculated_host_listings_count",
        "log_reviews_per_month"
    ],
    errors="ignore"
)

df_ohne_host_reviews.to_csv(
    "listings_cleaned4Decision_encoded_ohne_host_reviews.csv",
    index=False
)

print("Neue Datei wurde erstellt:")
print("listings_cleaned4Decision_encoded_ohne_host_reviews.csv")
print("Anzahl Spalten vorher:", df.shape[1])
print("Anzahl Spalten nachher:", df_ohne_host_reviews.shape[1])

Neue Datei wurde erstellt:
listings_cleaned4Decision_encoded_ohne_host_reviews.csv
Anzahl Spalten vorher: 220
Anzahl Spalten nachher: 218


In [11]:
#2.1
df = pd.read_csv("listings_cleaned4Decision_encoded_ohne_host_reviews.csv")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

#2.2
#2.2.1
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#2.2.2
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [16]:
#2.3
#2.3.1 
tree = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=15,
    random_state=42
)

tree.fit(X_train, y_train)

#2.3.2
val_predictions = tree.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)

print("Validation R²:", val_r2)

#2.3.3
test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)

print("Test R²:", test_r2)

Validation R²: 0.11779155458398871
Test R²: 0.10076373717870735


In [17]:
#2.4
#2.4.2
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                              Feature  Importance
4                                    instant_bookable    0.481780
2                                      minimum_nights    0.169585
3                                    availability_365    0.160992
5                                     price_per_guest    0.106391
6                         price_to_neighborhood_ratio    0.029497
1                                               price    0.021002
149        neighbourhood_group_cleansed_Reinickendorf    0.020106
194         property_type_Private room in rental unit    0.010646
0                                        accommodates    0.000000
9               neighbourhood_cleansed_Alexanderplatz    0.000000
10             neighbourhood_cleansed_Allende-Viertel    0.000000
11                neighbourhood_cleansed_Alt  Treptow    0.000000
12   neighbourhood_cleansed_Alt-Hohenschönhausen Nord    0.000000
13    neighbourhood_cleansed_Alt-Hohenschönhausen Süd    0.000000
14        

Entscheidung: alle mit neighbourhood ignorieren.

In [28]:
#1.1
df = pd.read_csv("listings_cleaned4Decision_encoded_ohne_host_reviews.csv")
df_ohne_neighbourhood = df.drop(
    columns=[col for col in df.columns if col.startswith("neighbourhood_")],
    errors="ignore"
)

df_ohne_neighbourhood.to_csv(
    "listings_cleaned4Decision_encoded_host_reviews_neighbourhood.csv",
    index=False
)

print("Neue Datei wurde erstellt:")
print("listings_cleaned4Decision_encoded_host_reviews_neighbourhood.csv")
print("Anzahl Spalten vorher:", df.shape[1])
print("Anzahl Spalten nachher:", df_ohne_neighbourhood.shape[1])

Neue Datei wurde erstellt:
listings_cleaned4Decision_encoded_host_reviews_neighbourhood.csv
Anzahl Spalten vorher: 218
Anzahl Spalten nachher: 71


In [29]:
#2.1
df = pd.read_csv("listings_cleaned4Decision_encoded_host_reviews_neighbourhood.csv")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

#2.2
#2.2.1
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#2.2.2
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [30]:
#2.3
#2.3.1 
tree = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=15,
    random_state=42
)

tree.fit(X_train, y_train)

#2.3.2
val_predictions = tree.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)

print("Validation R²:", val_r2)

#2.3.3
test_predictions = tree.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)

print("Test R²:", test_r2)

Validation R²: 0.12194370573659163
Test R²: 0.10590209880685475


In [31]:
#2.4
#2.4.2
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                      Feature  Importance
4                            instant_bookable    0.484529
2                              minimum_nights    0.185068
3                            availability_365    0.161911
5                             price_per_guest    0.106998
6                 price_to_neighborhood_ratio    0.029665
1                                       price    0.021122
47  property_type_Private room in rental unit    0.010707
0                                accommodates    0.000000
8                     property_type_Camper/RV    0.000000
7                          property_type_Boat    0.000000
10                         property_type_Cave    0.000000
11                         property_type_Dome    0.000000
12              property_type_Entire bungalow    0.000000
9               property_type_Casa particular    0.000000
14                property_type_Entire chalet    0.000000
15                 property_type_Entire condo    0.000000
16            

### 3. Random Forest ###

In [32]:
# 3.1 Daten laden
df = pd.read_csv("listings_cleaned4Decision_encoded.csv")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

# 3.2.1 Split 80/20
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3.2.2 Split 60/20/20
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [33]:
#3.3
#3.3.1
forest = RandomForestRegressor(
    n_estimators=100, #Anzahl Bäume
    max_depth=10, #Wie tief jeder einzelne Baum werden darf.
    min_samples_leaf=10, #Verhindert Overfitting
    random_state=42,
    n_jobs=-1 #Nutzt alle CPU-Kerne
)

forest.fit(X_train, y_train)

#3.3.2
predictions = forest.predict(X_test)

#3.3.3
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print("R²:", r2)
print("MAE:", mae)

R²: 0.2766830104739396
MAE: 0.14739270126039147


In [34]:
#3.4
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                               Feature  Importance
5                       calculated_host_listings_count    0.323059
8                                log_reviews_per_month    0.133478
3                                     availability_365    0.131668
6                                      price_per_guest    0.103811
2                                       minimum_nights    0.081566
1                                                price    0.058474
7                          price_to_neighborhood_ratio    0.056638
4                                     instant_bookable    0.032736
171                   property_type_Entire rental unit    0.015714
0                                         accommodates    0.013908
148                 neighbourhood_group_cleansed_Mitte    0.011735
155    neighbourhood_group_cleansed_Treptow - Köpenick    0.005607
145  neighbourhood_group_cleansed_Friedrichshain-Kr...    0.004886
217                             room_type_Private room    0.00

auf Datei ohne calculated_host_listings_count und log_reviews_per_month

In [35]:
# 3.1 Daten laden
df = pd.read_csv("listings_cleaned4Decision_encoded_ohne_host_reviews.csv")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

# 3.2.1 Split 80/20
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3.2.2 Split 60/20/20
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [36]:
#3.3
#3.3.1
forest = RandomForestRegressor(
    n_estimators=100, #Anzahl Bäume
    max_depth=10, #Wie tief jeder einzelne Baum werden darf.
    min_samples_leaf=10, #Verhindert Overfitting
    random_state=42,
    n_jobs=-1 #Nutzt alle CPU-Kerne
)

forest.fit(X_train, y_train)

#3.3.2
predictions = forest.predict(X_test)

#3.3.3
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print("R²:", r2)
print("MAE:", mae)

R²: 0.1620983201395586
MAE: 0.1597687018079519


In [37]:
#3.4
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                               Feature  Importance
4                                     instant_bookable    0.243897
3                                     availability_365    0.185782
5                                      price_per_guest    0.129938
2                                       minimum_nights    0.118881
6                          price_to_neighborhood_ratio    0.102811
1                                                price    0.091239
0                                         accommodates    0.021745
169                   property_type_Entire rental unit    0.020982
146                 neighbourhood_group_cleansed_Mitte    0.011627
143  neighbourhood_group_cleansed_Friedrichshain-Kr...    0.007510
194          property_type_Private room in rental unit    0.007075
215                             room_type_Private room    0.006513
149         neighbourhood_group_cleansed_Reinickendorf    0.005556
202                        property_type_Room in hotel    0.00

auf Datei ohne Neighbourhood

In [38]:
# 3.1 Daten laden
df = pd.read_csv("listings_cleaned4Decision_encoded_host_reviews_neighbourhood.csv")

# Zielvariable
y = df["review_scores_rating"]

# Features
X = df.drop("review_scores_rating", axis=1)

# 3.2.1 Split 80/20
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3.2.2 Split 60/20/20
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

In [39]:
#3.3
#3.3.1
forest = RandomForestRegressor(
    n_estimators=100, #Anzahl Bäume
    max_depth=10, #Wie tief jeder einzelne Baum werden darf.
    min_samples_leaf=10, #Verhindert Overfitting
    random_state=42,
    n_jobs=-1 #Nutzt alle CPU-Kerne
)

forest.fit(X_train, y_train)

#3.3.2
predictions = forest.predict(X_test)

#3.3.3
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print("R²:", r2)
print("MAE:", mae)

R²: 0.15263819674343937
MAE: 0.1614729525695077


In [40]:
#3.4
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(20))

                                            Feature  Importance
4                                  instant_bookable    0.242415
3                                  availability_365    0.199470
5                                   price_per_guest    0.139763
2                                    minimum_nights    0.124944
6                       price_to_neighborhood_ratio    0.117090
1                                             price    0.099653
0                                      accommodates    0.024492
22                 property_type_Entire rental unit    0.021273
47        property_type_Private room in rental unit    0.007798
68                           room_type_Private room    0.006949
55                      property_type_Room in hotel    0.005929
66                        room_type_Entire home/apt    0.004629
69                            room_type_Shared room    0.001414
23          property_type_Entire serviced apartment    0.000985
38              property_type_Private ro

### 4. XG Boost ###

In [42]:
#4.1
df = pd.read_csv("listings_cleaned4Decision_encoded.csv")

y = df["review_scores_rating"]

X = df.drop(
    "review_scores_rating",
    axis=1
)
#4.2
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

4.3
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

#4.3.1 Trainieren
xgb_model.fit(X_train, y_train)

#4.3.2 Validation
val_predictions = xgb_model.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)
val_mae = mean_absolute_error(y_val, val_predictions)

print("Validation R²:", val_r2)
print("Validation MAE:", val_mae)

#4.3.3 Test
test_predictions = xgb_model.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)

print("Test R²:", test_r2)
print("Test MAE:", test_mae)

#4.4 Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance.head(20))

Validation R²: 0.27629947265541976
Validation MAE: 0.1542316992591965
Test R²: 0.29030487008252426
Test MAE: 0.14432867536851932
                                               Feature  Importance
4                                     instant_bookable    0.045398
5                       calculated_host_listings_count    0.034770
74                         neighbourhood_cleansed_MV 2    0.031066
124               neighbourhood_cleansed_Teltower Damm    0.027084
61         neighbourhood_cleansed_Karl-Marx-Allee-Nord    0.024473
204                        property_type_Room in hotel    0.019937
94                        neighbourhood_cleansed_Ost 1    0.018663
126        neighbourhood_cleansed_Tempelhofer Vorstadt    0.017889
2                                       minimum_nights    0.016678
31                       neighbourhood_cleansed_Buckow    0.014643
85              neighbourhood_cleansed_Neu Lichtenberg    0.014131
11               neighbourhood_cleansed_Alexanderplatz    0.013385


auf Datei ohne calculated_host_listings_count und log_reviews_per_month

In [43]:
#4.1
df = pd.read_csv("listings_cleaned4Decision_encoded_ohne_host_reviews.csv")

y = df["review_scores_rating"]

X = df.drop(
    "review_scores_rating",
    axis=1
)
#4.2
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

4.3
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

#4.3.1 Trainieren
xgb_model.fit(X_train, y_train)

#4.3.2 Validation
val_predictions = xgb_model.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)
val_mae = mean_absolute_error(y_val, val_predictions)

print("Validation R²:", val_r2)
print("Validation MAE:", val_mae)

#4.3.3 Test
test_predictions = xgb_model.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)

print("Test R²:", test_r2)
print("Test MAE:", test_mae)

#4.4 Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance.head(20))

Validation R²: 0.19071072103884978
Validation MAE: 0.1658257020671539
Test R²: 0.1962293348759303
Test MAE: 0.15594761212929334
                                               Feature  Importance
4                                     instant_bookable    0.076233
122               neighbourhood_cleansed_Teltower Damm    0.024409
29                       neighbourhood_cleansed_Buckow    0.023309
59         neighbourhood_cleansed_Karl-Marx-Allee-Nord    0.022775
202                        property_type_Room in hotel    0.019465
92                        neighbourhood_cleansed_Ost 1    0.018000
214                               room_type_Hotel room    0.016576
143  neighbourhood_group_cleansed_Friedrichshain-Kr...    0.014974
2                                       minimum_nights    0.014878
72                         neighbourhood_cleansed_MV 2    0.014454
178    property_type_Private room in bed and breakfast    0.013994
28                     neighbourhood_cleansed_Buchholz    0.013182
1

auf Datei ohne Nachbarschaft

In [44]:
#4.1
df = pd.read_csv("listings_cleaned4Decision_encoded_host_reviews_neighbourhood.csv")

y = df["review_scores_rating"]

X = df.drop(
    "review_scores_rating",
    axis=1
)
#4.2
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42
)

4.3
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

#4.3.1 Trainieren
xgb_model.fit(X_train, y_train)

#4.3.2 Validation
val_predictions = xgb_model.predict(X_val)

val_r2 = r2_score(y_val, val_predictions)
val_mae = mean_absolute_error(y_val, val_predictions)

print("Validation R²:", val_r2)
print("Validation MAE:", val_mae)

#4.3.3 Test
test_predictions = xgb_model.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)

print("Test R²:", test_r2)
print("Test MAE:", test_mae)

#4.4 Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance.head(20))

Validation R²: 0.1588032855663324
Validation MAE: 0.16806976838115817
Test R²: 0.16606255824341976
Test MAE: 0.1593453413823929
                                            Feature  Importance
4                                  instant_bookable    0.174592
55                      property_type_Room in hotel    0.046216
2                                    minimum_nights    0.041589
42               property_type_Private room in home    0.035110
31  property_type_Private room in bed and breakfast    0.033567
69                            room_type_Shared room    0.032775
38              property_type_Private room in condo    0.032772
25               property_type_Entire vacation home    0.030477
47        property_type_Private room in rental unit    0.029860
43             property_type_Private room in hostel    0.029324
3                                  availability_365    0.029109
53             property_type_Room in boutique hotel    0.027633
67                             room_type

### 5. Ergebnisse insgesamt ###

1. Entscheidungen im Vorgehen: 
review_scores_rating wurde als Zielvariable definiert

Ausschluss der Variablen "id","listing_url","review_scores_accuracy","review_scores_cleanliness","review_scores_checkin","review_scores_communication","review_scores_location","review_scores_value","reviews_per_month","number_of_reviews"

im Moment durch Probieren wurde max_depth und min_samples_leaf festgelegt

random_state auf 42 festgelegt, für Reproduzierbarkeit

Probieren auch ohne calculated_host_listings_count und log_reviews_per_month, da sie sehr hohe Feature Importance haben und der Rest nur sehr wenig --> diese können aber nur sehr wenig beeinflusst werden durch Host, daher wäre es doof, wenn diese das einzige Ergebnis bleiben

Nachbarschaft raus, weil viele Variablen, die alle nichts erklären bzw nahezu nichts

Feature-Importance als Interpretationsmethode


### 6. weitere Schritte ###

1. Einmal nur auf den Nachbarschaften laufen lassen
2. Einmal nur auf den Nachbarschaften Gruppen laufen lassen

In [45]:
df = pd.read_csv("listings_cleaned4Decision_encoded.csv")

target = "review_scores_rating"

neighbourhood_cleansed_cols = [
    col for col in df.columns
    if col.startswith("neighbourhood_cleansed_")
]

neighbourhood_group_cols = [
    col for col in df.columns
    if col.startswith("neighbourhood_group_cleansed_")
]

df_neighbourhood_cleansed = df[neighbourhood_cleansed_cols + [target]]
df_neighbourhood_group = df[neighbourhood_group_cols + [target]]

df_neighbourhood_cleansed.to_csv(
    "only_neighbourhood_cleansed.csv",
    index=False
)

df_neighbourhood_group.to_csv(
    "only_neighbourhood_group_cleansed.csv",
    index=False
)

print("Dateien wurden erstellt.")
print("Anzahl neighbourhood_cleansed-Spalten:", len(neighbourhood_cleansed_cols))
print("Anzahl neighbourhood_group-Spalten:", len(neighbourhood_group_cols))

Dateien wurden erstellt.
Anzahl neighbourhood_cleansed-Spalten: 135
Anzahl neighbourhood_group-Spalten: 12


In [46]:
#6.1
def run_all_models(file_path):
    
    print("\n===================================")
    print("Datei:", file_path)
    print("===================================")
    
    df = pd.read_csv(file_path)
    
    y = df["review_scores_rating"]
    X = df.drop("review_scores_rating", axis=1)
    
    # 60-20-20 Split
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=0.25,
        random_state=42
    )
    
    models = {
        "Decision Tree": DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=10,
            random_state=42
        ),
        
        "Random Forest": RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ),
        
        "XGBoost": XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    }
    
    for model_name, model in models.items():
        
        print("\n---", model_name, "---")
        
        model.fit(X_train, y_train)
        
        val_predictions = model.predict(X_val)
        test_predictions = model.predict(X_test)
        
        val_r2 = r2_score(y_val, val_predictions)
        test_r2 = r2_score(y_test, test_predictions)
        
        val_mae = mean_absolute_error(y_val, val_predictions)
        test_mae = mean_absolute_error(y_test, test_predictions)
        
        print("Validation R²:", round(val_r2, 4))
        print("Test R²:", round(test_r2, 4))
        print("Validation MAE:", round(val_mae, 4))
        print("Test MAE:", round(test_mae, 4))
        
        importance = pd.DataFrame({
            "Feature": X.columns,
            "Importance": model.feature_importances_
        }).sort_values(by="Importance", ascending=False)
        
        print("\nTop 15 Feature Importances:")
        print(importance.head(15))

In [48]:
#6.1
run_all_models("only_neighbourhood_cleansed.csv")


Datei: only_neighbourhood_cleansed.csv

--- Decision Tree ---
Validation R²: -0.0114
Test R²: 0.002
Validation MAE: 0.1964
Test MAE: 0.1825

Top 15 Feature Importances:
                                              Feature  Importance
65                        neighbourhood_cleansed_MV 2    0.155502
134       neighbourhood_cleansed_südliche Luisenstadt    0.133314
57              neighbourhood_cleansed_Kurfürstendamm    0.095482
34                   neighbourhood_cleansed_Friedenau    0.094708
22                      neighbourhood_cleansed_Buckow    0.090993
74                 neighbourhood_cleansed_Moabit West    0.089641
120            neighbourhood_cleansed_Wedding Zentrum    0.088561
84               neighbourhood_cleansed_Osloer Straße    0.088277
2               neighbourhood_cleansed_Alexanderplatz    0.084153
83             neighbourhood_cleansed_Oberschöneweide    0.079368
5    neighbourhood_cleansed_Alt-Hohenschönhausen Nord    0.000000
1                 neighbourhood_cleans

In [49]:
#6.2
run_all_models("only_neighbourhood_group_cleansed.csv")


Datei: only_neighbourhood_group_cleansed.csv

--- Decision Tree ---
Validation R²: -0.0032
Test R²: -0.0089
Validation MAE: 0.1964
Test MAE: 0.1844

Top 15 Feature Importances:
                                              Feature  Importance
4                  neighbourhood_group_cleansed_Mitte    0.506098
0   neighbourhood_group_cleansed_Charlottenburg-Wilm.    0.199382
3   neighbourhood_group_cleansed_Marzahn - Hellers...    0.088745
7          neighbourhood_group_cleansed_Reinickendorf    0.064137
10  neighbourhood_group_cleansed_Tempelhof - Schön...    0.061937
9   neighbourhood_group_cleansed_Steglitz - Zehlen...    0.025966
5               neighbourhood_group_cleansed_Neukölln    0.019477
2            neighbourhood_group_cleansed_Lichtenberg    0.018232
11    neighbourhood_group_cleansed_Treptow - Köpenick    0.015226
6                 neighbourhood_group_cleansed_Pankow    0.000801
1   neighbourhood_group_cleansed_Friedrichshain-Kr...    0.000000
8                neighbourhood

Ergebnisse: 